# Cài đặt RAG

## Cài đặt

In [1]:
!pip install langchain-qdrant
!pip install accelerate bitsandbytes
!pip install bert_score
!pip install xformers==0.0.28.post3 --index-url https://download.pytorch.org/whl/cu121
!pip install optimum qdrant-client wikipedia FastAPI uvicorn pyngrok
!pip install --upgrade pydantic
!pip install vllm
!pip install ragas
!pip install -U \
      python-dotenv \
      langchain \
      langchain_openai \
      langchain_community \
      langchain-huggingface \
      langchain-google-genai \
      streamlit \
      faiss-cpu \
      sentence-transformers \
      pypdf \
      docx2txt\
      vllm\
      triton
!pip install autoawq
!pip install accelerate transformers

Looking in indexes: https://download.pytorch.org/whl/cu121
  Using cached triton-3.2.0-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (1.4 kB)


## Khởi tạo biến môi trường

In [ ]:
#Khai báo khóa API key
from google.colab import userdata
import os
os.environ["GOOGLE_API_KEY"]  = userdata.get("GOOGLE_API_KEY")

In [ ]:
# Cài đặt thông tin truy cập vector database
QDRANT_API_KEY = userdata.get("QDRANT_API_KEY")
QDRANT_URL = userdata.get("QDRANT_URL")

EMBEDDINGS_MODEL_NAME="intfloat/multilingual-e5-base"
HUGGINGFACE_API_KEY= userdata.get("HUGGINGFACE_API_KEY")
QDRANT_COLLECTION_NAME = "ITUS_e5_600"
GENERATE_MODEL_NAME="AITeamVN/Vi-Qwen2-3B-RAG"3B-RAG"

## RAG gemini

In [4]:
!pip install pyvi sentence-transformers nltk rouge-score

  Using cached pyvi-0.1.1-py2.py3-none-any.whl.metadata (2.5 kB)
  Using cached rouge_score-0.1.2.tar.gz (17 kB)
  Preparing metadata (setup.py) ... done
  Using cached sklearn_crfsuite-0.5.0-py2.py3-none-any.whl.metadata (4.9 kB)
  Using cached python_crfsuite-0.9.11-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.3 kB)
Using cached pyvi-0.1.1-py2.py3-none-any.whl (8.5 MB)
Using cached sklearn_crfsuite-0.5.0-py2.py3-none-any.whl (10 kB)
Using cached python_crfsuite-0.9.11-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.3 MB)
  Created wheel for rouge-score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=0e31282d13bfc08ae2707975d66e084882f097ecae6adb1522f18e127027cc85
  Stored in directory: /root/.cache/pip/wheels/1e/19/43/8a442dc83660ca25e163e1bd1f89919284ab0d0c1475475148
Successfully built rouge-score


In [5]:
import nltk
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [6]:
from langchain.schema.document import Document
from langchain.schema.retriever import BaseRetriever
from langchain.retrievers import WikipediaRetriever
from langchain_qdrant import QdrantVectorStore
from langchain.llms import HuggingFacePipeline
from qdrant_client import QdrantClient
from langchain.prompts import PromptTemplate
from langchain.embeddings import HuggingFaceInferenceAPIEmbeddings
from langchain.chains import RetrievalQA, MultiRetrievalQAChain
from langchain.llms import VLLM
from langchain.llms import HuggingFaceHub
from typing import List
import asyncio
from pydantic import BaseModel, Field
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from asyncio import to_thread
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.chains import ConversationalRetrievalChain
import pandas as pd
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
from pyvi import ViTokenizer
from nltk.translate.meteor_score import meteor_score
from sentence_transformers import SentenceTransformer, util
import time

class LLMServe:
    def __init__(self, model_name=GENERATE_MODEL_NAME) -> None:
        self.embeddings = self.load_embeddings()
        self.current_source = "qdrant"
        self.retriever = self.load_retriever(retriever_name=self.current_source, embeddings=self.embeddings)
        self.pipe = self.load_model_pipeline(model_name=model_name, max_new_tokens=512)
        self.prompt = self.load_prompt_template()
        self.sbert_model = SentenceTransformer("sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
        self.rag_pipeline = self.load_rag_pipeline(
            llm=self.pipe,
            retriever=self.retriever,
            prompt=self.prompt
        )

    def load_embeddings(self):
        embeddings = HuggingFaceInferenceAPIEmbeddings(
            model_name=EMBEDDINGS_MODEL_NAME,
            api_key=HUGGINGFACE_API_KEY,
        )
        return embeddings

    def load_retriever(self, retriever_name, embeddings):
        """
        Load and create appropriate retriever based on retriever_name
        """
        base_retriever = None
        if retriever_name == "wiki":
            base_retriever = WikipediaRetriever(
                lang="vi",
                doc_content_chars_max=800,
                top_k_results=10
            )
        elif retriever_name == "qdrant":
            client = QdrantClient(
                url=QDRANT_URL,
                api_key=QDRANT_API_KEY,
                prefer_grpc=False
            )

            db = QdrantVectorStore(
                client=client,
                embedding=embeddings,
                collection_name=QDRANT_COLLECTION_NAME
            )
            base_retriever = db.as_retriever(search_kwargs={"k": 10})

        return base_retriever

    def load_model_pipeline(self, model_name="gemini", max_new_tokens=512):
        """
        Load and create appropriate model pipeline based on model_name
        """
        if model_name == "gemini":
            llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash")
        else:
            llm = VLLM(
                model=model_name,
                trust_remote_code=True,
                max_new_tokens=max_new_tokens,
                top_k=10,
                top_p=0.95,
                temperature=0.4,
                dtype="half",
            )
        return llm

    def load_prompt_template(self):


        query_template = '''Thực hiện trả lời câu hỏi từ thông tin có trong ngữ cảnh được cho. Chú ý các yêu cầu sau:
                    - Câu trả lời phải chính xác, ngắn gọn và đúng trọng tâm câu hỏi.
                    - Chỉ sử dụng các thông tin có trong ngữ cảnh được cung cấp.
                    - Nếu ngữ cảnh không chứa thông tin về câu trả lời thì từ chối trả lời, không suy luận gì thêm
                    - Khi từ chối trả lời, cung cấp thông tin liên lạc của khoa như sau để người dùng liên lạc:
                    "Vui lòng liên lạc Khoa Công Nghệ Thông Tin, trường Đại học Khoa Học Tự Nhiên - Đại học Quốc Gia TP.Hồ Chí Minh để giải đáp:
                    Địa chỉ: Phòng I.54, toà nhà I, 227 Nguyễn Văn Cừ, Q.5, TP.HCM
                    Điện thoại: (028) 62884499
                    Email: info@fit.hcmus.edu.vn"
                    ### Ngữ cảnh :
                    {context}

                    ### Câu hỏi :
                    {question}

                    ### Trả lời :'''
        prompt = PromptTemplate(template=query_template, input_variables=["context", "question"])
        return prompt

    def load_rag_pipeline(self, llm, retriever, prompt):
        rag_pipeline = RetrievalQA.from_chain_type(
            llm=llm,
            chain_type='stuff',
            retriever=retriever,
            chain_type_kwargs={
                "prompt": prompt
            },
            return_source_documents=True,
        )
        return rag_pipeline

    def rag(self, source):
        if source == self.current_source:
            return self.rag_pipeline
        else:
            self.retriever = self.load_retriever(retriever_name=source, embeddings=self.embeddings)
            self.rag_pipeline = self.load_rag_pipeline(
                llm=self.pipe,
                retriever=self.retriever,
                prompt=self.prompt
            )
            self.current_source = source
            return self.rag_pipeline


    def calculate_bleu(self, generated_tokens, reference_tokens):
        """
        Calculate BLEU score for a single generated answer and reference.
        :param generated_tokens: List of generated tokens.
        :param reference_tokens: List of reference tokens.
        :return: BLEU score.
        """
        # Use NLTK's BLEU score implementation with smoothing
        smoothie = SmoothingFunction().method4  # Apply smoothing
        bleu_score = sentence_bleu([reference_tokens], generated_tokens, smoothing_function=smoothie)
        return bleu_score

    def calculate_rouge(self, generated_tokens, reference_tokens):
        """
        Calculate ROUGE scores for a single generated answer and reference.
        :param generated_tokens: List of generated tokens.
        :param reference_tokens: List of reference tokens.
        :return: A dictionary with ROUGE-1, ROUGE-2, and ROUGE-L scores.
        """
        scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
        scores = scorer.score(" ".join(reference_tokens), " ".join(generated_tokens))
        return scores


    def evaluate_generation(self, input_filepath: str, output_filepath: str, delay: float = 5.0):
        """
        Evaluate the generated answers using BLEU, ROUGE, METEOR, and SBERT cosine similarity.
        Includes a delay between requests to limit API call frequency.

        :param input_filepath: Path to the input Excel file containing questions and expected outputs.
        :param output_filepath: Path to save the evaluation results as an Excel file.
        :param delay: Time in seconds to wait between processing each question.
        """
        # Load data từ file Excel
        data = pd.read_excel(input_filepath)
        questions = data['question'].tolist()
        expected_outputs = data['expected_output'].tolist()

        results = []

        bleu_scores = []
        rouge_1_scores = []
        rouge_2_scores = []
        rouge_l_scores = []
        meteor_scores = []
        sbert_similarities = []

        for idx, question in enumerate(questions):
            print(f"Processing question {idx + 1}/{len(questions)}: {question}")

            try:
                # Gọi pipeline để lấy câu trả lời
                response = self.rag_pipeline({"query": question})
                generated_answer = response.get("result", "")

                # Tokenize văn bản sử dụng ViTokenizer
                generated_answer_tokens = ViTokenizer.tokenize(generated_answer).split()
                expected_output_tokens = ViTokenizer.tokenize(expected_outputs[idx]).split()

                # Tính BLEU và ROUGE
                bleu_score = self.calculate_bleu(generated_answer_tokens, expected_output_tokens)
                rouge_scores = self.calculate_rouge(generated_answer_tokens, expected_output_tokens)

                # Tính METEOR
                meteor = meteor_score([expected_output_tokens], generated_answer_tokens)

                # Tính SBERT Cosine Similarity
                generated_embedding = self.sbert_model.encode(generated_answer, convert_to_tensor=True)
                expected_embedding = self.sbert_model.encode(expected_outputs[idx], convert_to_tensor=True)
                sbert_similarity = util.pytorch_cos_sim(generated_embedding, expected_embedding).item()

                # Lưu điểm số để tính trung bình
                bleu_scores.append(bleu_score)
                rouge_1_scores.append(rouge_scores["rouge1"].fmeasure)
                rouge_2_scores.append(rouge_scores["rouge2"].fmeasure)
                rouge_l_scores.append(rouge_scores["rougeL"].fmeasure)
                meteor_scores.append(meteor)
                sbert_similarities.append(sbert_similarity)

                # Lưu kết quả
                results.append({
                    "question": question,
                    "generated_answer": generated_answer,
                    "expected_output": expected_outputs[idx],
                    "bleu_score": bleu_score,
                    "rouge_1": rouge_scores["rouge1"].fmeasure,
                    "rouge_2": rouge_scores["rouge2"].fmeasure,
                    "rouge_L": rouge_scores["rougeL"].fmeasure,
                    "meteor": meteor,
                    "sbert_similarity": sbert_similarity
                })
            except Exception as e:
                print(f"Error processing question {idx + 1}: {e}")
                results.append({
                    "question": question,
                    "generated_answer": None,
                    "expected_output": expected_outputs[idx],
                    "bleu_score": None,
                    "rouge_1": None,
                    "rouge_2": None,
                    "rouge_L": None,
                    "meteor": None,
                    "sbert_similarity": None
                })

            # Thêm delay giữa các yêu cầu
            time.sleep(delay)

        # Tính trung bình các độ đo
        avg_bleu = sum(bleu_scores) / len(bleu_scores) if bleu_scores else 0
        avg_rouge_1 = sum(rouge_1_scores) / len(rouge_1_scores) if rouge_1_scores else 0
        avg_rouge_2 = sum(rouge_2_scores) / len(rouge_2_scores) if rouge_2_scores else 0
        avg_rouge_l = sum(rouge_l_scores) / len(rouge_l_scores) if rouge_l_scores else 0
        avg_meteor = sum(meteor_scores) / len(meteor_scores) if meteor_scores else 0
        avg_sbert = sum(sbert_similarities) / len(sbert_similarities) if sbert_similarities else 0

        print("\nAverage Scores:")
        print(f"Average BLEU: {avg_bleu:.4f}")
        print(f"Average ROUGE-1: {avg_rouge_1:.4f}")
        print(f"Average ROUGE-2: {avg_rouge_2:.4f}")
        print(f"Average ROUGE-L: {avg_rouge_l:.4f}")
        print(f"Average METEOR: {avg_meteor:.4f}")
        print(f"Average SBERT Similarity: {avg_sbert:.4f}")

        # Convert results to DataFrame và lưu vào file Excel
        results_df = pd.DataFrame(results)
        results_df.to_excel(output_filepath, index=False)
        print(f"Evaluation results saved to {output_filepath}")


In [7]:
# Có thể thay đổi model khác bằng cách đổi tên model
#Ví dụ:
model_name = GENERATE_MODEL_NAME
app = LLMServe(model_name=model_name)

INFO 01-28 01:28:07 __init__.py:183] Automatically detected platform cuda.


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/709 [00:00<?, ?B/s]

WARNING 01-28 01:28:11 config.py:2318] Casting torch.bfloat16 to torch.float16.
INFO 01-28 01:28:23 config.py:520] This model supports multiple tasks: {'embed', 'generate', 'reward', 'classify', 'score'}. Defaulting to 'generate'.
INFO 01-28 01:28:23 llm_engine.py:232] Initializing an LLM engine (v0.7.0) with config: model='AITeamVN/Vi-Qwen2-3B-RAG', speculative_config=None, tokenizer='AITeamVN/Vi-Qwen2-3B-RAG', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, override_neuron_config=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=32768, download_dir=None, load_format=LoadFormat.AUTO, tensor_parallel_size=1, pipeline_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, kv_cache_dtype=auto,  device_config=cuda, decoding_config=DecodingConfig(guided_decoding_backend='xgrammar'), observability_config=ObservabilityConfig(otlp_traces_endpoint=None, collect_model_forward_time=False, collect_model_execut

tokenizer_config.json:   0%|          | 0.00/5.25k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/471M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [9]:
input_filepath = "/content/QA_data.xlsx"
output_filepath = "/content/evaluation_results.xlsx"

# Thực hiện đánh giá
app.evaluate_generation(input_filepath, output_filepath=output_filepath)

Processing question 1/144: Tổng số tín chỉ bắt buộc của ngành Hệ thống thông tin là bao nhiêu?


<ipython-input-6-4b7f8f512d54>:195: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 1.0. Use :meth:`~invoke` instead.
  response = self.rag_pipeline({"query": question})
Processed prompts: 100%|██████████| 1/1 [00:15<00:00, 15.59s/it, est. speed input: 155.71 toks/s, output: 32.86 toks/s]


Processing question 2/144: Tên văn bằng ngành Khoa học máy tính sau khi tốt nghiệp là gì?


Processed prompts: 100%|██████████| 1/1 [00:15<00:00, 15.51s/it, est. speed input: 141.72 toks/s, output: 33.03 toks/s]


Processing question 3/144: Sinh viên phải làm gì để đăng ký học phần đầu mỗi học kỳ?


...

Processing question 143/144: Sinh viên được yêu cầu về kỹ năng ngoại ngữ nào?


Processed prompts: 100%|██████████| 1/1 [00:15<00:00, 15.55s/it, est. speed input: 117.41 toks/s, output: 32.92 toks/s]


Processing question 144/144: Chuyên ngành Mạng máy tính và Viễn thông yêu cầu tích lũy bao nhiêu tín chỉ tự chọn chuyên ngành?


Processed prompts: 100%|██████████| 1/1 [00:15<00:00, 15.76s/it, est. speed input: 146.87 toks/s, output: 32.50 toks/s]



Average Scores:
Average BLEU: 0.0614
Average ROUGE-1: 0.2397
Average ROUGE-2: 0.1992
Average ROUGE-L: 0.2156
Average METEOR: 0.3339
Average SBERT Similarity: 0.8032
Evaluation results saved to /content/evaluation_results.xlsx
